# Harmony4D Drive Benchmark → Google Sheets (Colab)

Chạy `run_drive_benchmark.py` trên Google Colab và xuất kết quả trực tiếp ra **Google Sheets** (không xuất CSV lên Drive):
- **Tự động ghim dòng tiêu đề** (`freeze(rows=1)`) + định dạng bảng qua `brute_force_runner.decorate`.
- **Tách riêng từng file Google Sheet cho từng người** (ví dụ `aria01` và `aria02` ra **2 file riêng biệt**), tự động đồng bộ mỗi 60 giây và hỗ trợ resume trực tiếp từ Google Sheets khi ngắt runtime.

In [ ]:
#@title 1. Cài môi trường, clone repo, tải SMPL_NEUTRAL.pkl và cấu hình Multi-Person
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
BRANCH = "experiment/observability-aware-trust-region"  #@param {type:"string"}
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
FUSION_METHOD = "partwise"  #@param ["partwise", "aotr", "naive"]
MULTI_PERSON = True  #@param {type:"boolean"}
STATIC_CAMERA = True  #@param {type:"boolean"}
CONFIGS = ""  #@param {type:"string"}
REVERSE = True  #@param {type:"boolean"}
WORKERS = 0  #@param {type:"integer"}
SHEET_PREFIX = ""  #@param {type:"string"}
DRIVE_FOLDER_ID = ""  #@param {type:"string"}
PUBLIC_READ = True  #@param {type:"boolean"}

REPO_DIR = Path("/content/optimization_model_monocular_3")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-f", "-B", BRANCH, f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "chumpy"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "PyYAML", "gspread", "numpy<2.3"], check=True)

smpl_path = REPO_DIR / "models" / "SMPL_NEUTRAL.pkl"
if not smpl_path.exists() or smpl_path.stat().st_size < 1_000_000:
    import gdown
    smpl_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(smpl_path), quiet=False)

import yaml
from fusion_pipeline.aotr_benchmark import report_csv_name

cfg_path = REPO_DIR / "configs" / "pipeline.yml"
cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
cfg["fusion"]["method"] = FUSION_METHOD
cfg["fusion"].setdefault("occlusion", {})["multi_person"] = bool(MULTI_PERSON)
cfg["fusion"].setdefault("partwise", {}).setdefault("multi_person", {})["enabled"] = bool(MULTI_PERSON)
cfg["fusion"]["partwise"]["multi_person"]["static_camera_orientation"] = bool(MULTI_PERSON and STATIC_CAMERA)
cfg["fusion"].setdefault("aotr", {})["temporal_alignment_fallback"] = bool(MULTI_PERSON)
cfg["fusion"]["aotr"]["max_cross_view_bone_angle_deg"] = 50.0 if MULTI_PERSON else None
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")

REPORT_NAME = report_csv_name(cfg)
REPORT_TAG = Path(REPORT_NAME).stem.replace("brute_force_report_", "")
print(f"Ready | branch={BRANCH} | method={FUSION_METHOD} | multi_person={MULTI_PERSON} | static_cam={STATIC_CAMERA} | tag={REPORT_TAG} | cpus={os.cpu_count()}")

In [ ]:
#@title 2. Chạy benchmark & ghi trực tiếp ra Google Sheets (2 người = 2 file riêng, ghim tiêu đề)
import csv, os, subprocess, sys, threading
from pathlib import Path
import gspread, yaml
from google.colab import auth
from google.auth import default
from brute_force_runner import decorate
from run_drive_benchmark import is_drive_segment

auth.authenticate_user()
creds, _ = default()
GC = gspread.authorize(creds)
SHEET_URLS = {}
_LAST_SYNCED = {}

def sheet_title(seq_name, person):
    return f"{SHEET_PREFIX}{seq_name}_{person}_{REPORT_TAG}"

def open_or_create_sheet(title):
    try:
        sh = GC.open(title)
    except gspread.SpreadsheetNotFound:
        folder = DRIVE_FOLDER_ID.strip() or None
        try:
            sh = GC.create(title, folder_id=folder) if folder else GC.create(title)
        except TypeError:
            sh = GC.create(title)
        if PUBLIC_READ:
            try:
                sh.share("", perm_type="anyone", role="reader")
            except Exception:
                pass
        print(f"[GSheet] Tạo mới: {title} -> {sh.url}", flush=True)
    SHEET_URLS[title] = sh.url
    return sh

def write_person_sheet(title, header, person_rows, is_final=False):
    # ponytail: skip redundant API writes when row count/tail unchanged
    rows = [header, *person_rows]
    if is_final:
        rows.append(["End"] * len(header))
    sig = (len(rows), is_final, tuple(rows[-1]) if rows else ())
    if _LAST_SYNCED.get(title) == sig:
        return
    sh = open_or_create_sheet(title)
    ws = sh.sheet1
    ncols = len(header)
    if ws.row_count < len(rows) or ws.col_count < ncols:
        ws.resize(rows=max(ws.row_count, len(rows) + 20), cols=max(ws.col_count, ncols + 2))
    ws.clear()
    try:
        ws.update(values=rows, range_name="A1")
    except TypeError:
        ws.update(rows)
    decorate(ws, len(rows), ncols)  # freeze(rows=1) + header bold + zebra rows
    _LAST_SYNCED[title] = sig
    if is_final:
        print(f"[GSheet] Hoàn tất ({len(person_rows)} cặp, đã ghim tiêu đề): {title}\n  🔗 {sh.url}", flush=True)

def split_rows_by_person(report_path):
    if not report_path.is_file():
        return None, {}
    with report_path.open("r", encoding="utf-8-sig", newline="") as f:
        all_rows = list(csv.reader(f))
    if len(all_rows) < 2:
        return None, {}
    header = all_rows[0]
    tp_idx = header.index("Target Person") if "Target Person" in header else -1
    seg_idx = header.index("Segment") if "Segment" in header else -1
    proto_idx = header.index("Protocol") if "Protocol" in header else -1
    by_person = {}
    for row in all_rows[1:]:
        if not row or (proto_idx >= 0 and proto_idx < len(row) and row[proto_idx] == "End"):
            continue
        if tp_idx >= 0 and tp_idx < len(row) and row[tp_idx] not in ("", "End"):
            person = row[tp_idx].strip().lower()
        elif seg_idx >= 0 and seg_idx < len(row) and "_" in row[seg_idx]:
            person = row[seg_idx].rsplit("_", 1)[-1].strip().lower()
        else:
            person = "aria01"
        by_person.setdefault(person, []).append(row)
    return header, by_person

def restore_from_gsheets(seq_name, report_path, candidates=("aria01", "aria02", "aria03", "aria04")):
    """Pull existing per-person Google Sheets into local worker buffer so benchmark resumes cleanly."""
    header, combined = None, []
    for person in candidates:
        title = sheet_title(seq_name, person)
        try:
            sh = GC.open(title)
        except gspread.SpreadsheetNotFound:
            continue
        SHEET_URLS[title] = sh.url
        vals = sh.sheet1.get_all_values()
        if len(vals) < 2:
            continue
        if header is None:
            header = vals[0]
        p_idx = header.index("Protocol") if "Protocol" in header else -1
        rows = [r for r in vals[1:] if r and not (p_idx >= 0 and p_idx < len(r) and r[p_idx] == "End")]
        combined.extend(rows)
        print(f"[GSheet] Resume {len(rows)} cặp từ {title}", flush=True)
    if header and combined:
        report_path.parent.mkdir(parents=True, exist_ok=True)
        with report_path.open("w", encoding="utf-8-sig", newline="") as f:
            csv.writer(f).writerows([header, *combined])

def sync_seq_to_gsheets(seq_name, report_path, is_final=False):
    header, by_person = split_rows_by_person(report_path)
    if not header or not by_person:
        return
    for person, rows in sorted(by_person.items()):
        write_person_sheet(sheet_title(seq_name, person), header, rows, is_final=is_final)

out_root = REPO_DIR / "output" / "drive_benchmark"
workers = WORKERS if WORKERS > 0 else max(1, (os.cpu_count() or 2) - 1)
targets = [REPO_DIR / c for c in CONFIGS.split()] if CONFIGS.strip() else sorted((REPO_DIR / "configs").glob("*.yml"))
targets = [p for p in targets if any(is_drive_segment(s) for s in (yaml.safe_load(p.read_text(encoding="utf-8")) or {}).get("segments", []))]
if REVERSE:
    targets = list(reversed(targets))

env = {**os.environ, "PYTHONUTF8": "1", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1"}
for cfg_file in targets:
    seq_name = cfg_file.stem
    work_report = out_root / seq_name / REPORT_NAME
    restore_from_gsheets(seq_name, work_report)
    cmd = [sys.executable, "-u", "run_drive_benchmark.py", "--workers", str(workers), *(["--reverse"] if REVERSE else []), str(cfg_file)]
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stop_sync = threading.Event()
    def _sync_loop(seq=seq_name, rp=work_report):
        while not stop_sync.wait(60):
            try:
                sync_seq_to_gsheets(seq, rp, is_final=False)
            except Exception as exc:
                print(f"[GSheet] Sync warning ({seq}): {exc}", flush=True)
    t = threading.Thread(target=_sync_loop, daemon=True)
    t.start()
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    stop_sync.set()
    t.join()
    sync_seq_to_gsheets(seq_name, work_report, is_final=(proc.returncode == 0))
    if proc.returncode not in (0, 1):
        raise RuntimeError(f"{cfg_file.name} failed with exit code {proc.returncode}")

In [ ]:
#@title 3. Tổng hợp kết quả trực tiếp từ Google Sheets & kiểm toán Person Mapping
import glob, json
from pathlib import Path
import pandas as pd

def _num(s):
    return pd.to_numeric(s, errors="coerce")

rows = []
for title, url in sorted(SHEET_URLS.items()):
    vals = GC.open(title).sheet1.get_all_values()
    if len(vals) < 2:
        continue
    df = pd.DataFrame(vals[1:], columns=vals[0])
    df = df[df["Protocol"].astype(str) != "End"]
    if df.empty:
        continue
    for (seg, person), g in df.groupby(["Segment", "Target Person"]):
        pa_m, pa_f = _num(g["pa-mpjpe.all_baseline"]).mean(), _num(g["pa-mpjpe.all_method"]).mean()
        mp_m, mp_f = _num(g["mpjpe.all_baseline"]).mean(), _num(g["mpjpe.all_method"]).mean()
        rows.append({
            "Google Sheet": title, "Segment": seg, "Person": person, "Pairs": len(g),
            "PA mono": round(pa_m, 2), "PA fused": round(pa_f, 2), "dPA": round(pa_f - pa_m, 2),
            "MP mono": round(mp_m, 2), "MP fused": round(mp_f, 2), "dMP": round(mp_f - mp_m, 2),
            "MBLE": round(_num(g["mble.method"]).mean(), 2) if "mble.method" in g else None,
            "Acc mono": round(_num(g["accel.baseline"]).mean(), 2) if "accel.baseline" in g else None,
            "Acc fused": round(_num(g["accel.method"]).mean(), 2) if "accel.method" in g else None,
            "URL": url,
        })
display(pd.DataFrame(rows))

map_rows = []
for path in sorted(glob.glob(str(REPO_DIR / "output/drive_benchmark/*/person_labels_report.json"))):
    rep = json.loads(Path(path).read_text(encoding="utf-8"))
    seq = Path(path).parent.name
    for cam_id, info in sorted((rep.get("cameras") or {}).items()):
        map_rows.append({
            "Sequence": seq, "Camera": cam_id,
            "Mapping": info.get("mapping"),
            "Covered": info.get("frames_covered"),
            "Switch frames": info.get("switch_frames"),
        })
if map_rows:
    print("\n=== Person Mapping Audit ===")
    display(pd.DataFrame(map_rows))